# Отчёт по этапу 4 дипломного проекта

**Тема:** «Разработка интеллектуального Telegram-бота на основе языковой модели для консультирования по нормативной документации в области электроэнергетики (ПУЭ, ПТЭЭП, ГОСТ)» (проект Diplom_AI — RAG-система по нормативным документам).

**Этап 4:** создание прототипа и получение первой точности (для RAG: прототип рабочего контура и метрики Hit@k, MRR/nDCG, генерации и цитирования).

**Репозиторий:** https://github.com/kirag-ozyaz/Diplom_AI

---

Ноутбук читает только сохранённые файлы результатов из `scripts/stage4_eval/results/` (и `data/`); Milvus, Ollama и GPU не нужны. Корень репозитория определяется автоматически. Повторный прогон оценки (нужны Docker, Milvus, Ollama) — в разделе 6, по умолчанию выключен флагом `RUN_LIVE = False`.

Структура, нумерация разделов, таблиц и рисунков совпадают с отчётом Word этапа 4; все числа ноутбук считает из тех же файлов результатов.

Опыты пронумерованы с 1: опыт 1 — база (`diplom_multimodal`), опыт 2 — префиксы E5 (`diplom_multimodal_e5`), опыт 3 — E5 и реранкер (`diplom_multimodal_e5_rerank`). В файлах проекта те же опыты обозначены 0, 1, 2 (суффиксы JSON: нет / `_e5` / `_rerank`).

## 0. Подготовка: поиск корня проекта и вспомогательные функции

In [ ]:
# Флаг повторного прогона оценки (раздел 6). False — только чтение сохранённых результатов.
RUN_LIVE = False

import json, re
from collections import Counter
from pathlib import Path
import plotly.graph_objects as go
import numpy as np
from IPython.display import Markdown, display

def find_root(start=None):
    """Корень репозитория: первая папка вверх от текущей, где есть scripts/stage4_eval/results."""
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "scripts" / "stage4_eval" / "results").is_dir():
            return p
    raise FileNotFoundError("Не найден корень проекта (нет scripts/stage4_eval/results). "
                            "Откройте ноутбук внутри репозитория Diplom_AI.")

ROOT = find_root()
RES = ROOT / "scripts" / "stage4_eval" / "results"
RDIR, GDIR = RES / "retrieval", RES / "generation"
REPORTS = ROOT / "Этапы" / "Reports" / "etap4"
QUESTIONS = ROOT / "data" / "rag_eval_questions" / "eval_questions.json"

EXPERIMENTS = [  # (номер, подпись, суффикс hitk/gen/perf/judge, режим rank)
    (1, "Опыт 1 (база)", "", "base"),
    (2, "Опыт 2 (E5)", "_e5", "e5"),
    (3, "Опыт 3 (реранкер)", "_rerank", "rerank"),
]
LABELS = [e[1] for e in EXPERIMENTS]
COLORS = ["#7f7f7f", "#1f77b4", "#2ca02c"]

def load(p):
    p = Path(p)
    return json.loads(p.read_text(encoding="utf-8-sig")) if p.is_file() else None

def num(v, d=1):
    """Число в русском формате: 81.3 -> '81,3'."""
    if v is None:
        return "—"
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return f"{v:.{d}f}".replace(".", ",")
    return str(v)

def md_table(header, rows):
    out = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    out += ["| " + " | ".join(str(x) for x in r) + " |" for r in rows]
    display(Markdown("\n".join(out)))

def _show(fig, title, ylabel, ymax):
    """Общее оформление живого графика Plotly."""
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=460,
        margin=dict(t=88, b=48, l=56, r=24),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
        hovermode="x unified",
    )
    axis = dict(title=ylabel)
    if ymax is not None:
        axis["range"] = [0, ymax]
    fig.update_yaxes(**axis)
    fig.show()

def bar_chart(categories, series, title, ylabel, ymax=None, fmt="{:.1f}", colors=COLORS):
    """Сгруппированная столбчатая диаграмма: series = [(подпись, [значения])]."""
    fig = go.Figure()
    xs = [str(c) for c in categories]
    for i, (name, vals) in enumerate(series):
        y, text = [], []
        for v in vals:
            if v is None:
                y.append(None)
                text.append("")
            else:
                y.append(float(v))
                text.append(fmt.format(v).replace(".", ","))
        fig.add_trace(go.Bar(
            name=name, x=xs, y=y, text=text, textposition="outside", cliponaxis=False,
            marker_color=colors[i % len(colors)],
            hovertemplate="%{x}<br>%{fullData.name}: %{text}<extra></extra>",
        ))
    fig.update_layout(barmode="group")
    _show(fig, title, ylabel, ymax)

def line_chart(categories, series, title, ylabel, ymax=None, colors=None):
    """Линии с маркерами: series = [(подпись, [значения])]."""
    palette = colors or ["#1f77b4", "#ff7f0e", "#2ca02c"]
    fig = go.Figure()
    xs = [str(c) for c in categories]
    for i, (name, vals) in enumerate(series):
        fig.add_trace(go.Scatter(
            name=name, x=xs,
            y=[None if v is None else float(v) for v in vals],
            mode="lines+markers",
            line=dict(color=palette[i % len(palette)], width=2),
            marker=dict(size=8),
            hovertemplate="%{x}<br>%{fullData.name}: %{y}<extra></extra>",
        ))
    _show(fig, title, ylabel, ymax)

# Загрузка всех результатов трёх опытов
DATA = {}
for n, label, suf, mode in EXPERIMENTS:
    DATA[n] = {
        "hitk": load(RDIR / f"stage4_hitk_results{suf}.json"),
        "rank": load(RDIR / f"stage4_rank_results_{mode}.json"),
        "gen": load(GDIR / f"stage4_gen_eval_results{suf}.json"),
        "perf": load(GDIR / f"stage4_performance_metrics{suf}.json"),
        "judge": load(GDIR / f"stage4_llm_judge_metrics{suf}.json"),
    }
print("Корень проекта:", ROOT)
for n, label, suf, mode in EXPERIMENTS:
    have = [k for k, v in DATA[n].items() if v is not None]
    miss = [k for k, v in DATA[n].items() if v is None]
    print(f"{label}: есть {have}" + (f"; нет {miss}" if miss else ""))


## 1. Тема, описание задачи

**Задача проекта.** Пользователь задаёт вопрос на русском языке, система находит подходящие пункты нормативного документа и формирует ответ со ссылками на номера пунктов. Используется схема RAG: семантический поиск фрагментов и генерация ответа языковой моделью по найденным фрагментам.

**Задача этапа 4**:

1. собрать прототип рабочего контура: вопрос преобразуется в вектор, по нему в Milvus ищутся ближайшие фрагменты ПУЭ, первые k фрагментов передаются языковой модели, и она пишет ответ со ссылками на пункты ПУЭ;
2. подготовить контрольный набор вопросов с эталонными пунктами ПУЭ;
3. получить первую точность: Hit@1/3/5 для поиска, MRR и nDCG@k, метрики генерации и цитирования;
4. провести опыты по улучшению поиска и сравнить их на одном наборе вопросов.

Модели предобученные, обучение весов не выполнялось, поэтому графиков loss/accuracy по эпохам нет. «Первой точностью» считается Hit@k — доля вопросов, для которых эталонный пункт ПУЭ попал в первые k найденных фрагментов.

## 2. База данных

**Источник:** ПУЭ, 7-е издание, главы в DOCX (`data/raw/Нормативная база/ПУЭ/DOCX/`), из них получены Markdown (`data/extracted/`) и чанки JSONL (`data/chunked/`). Документ открытый, конфиденциальных данных нет. ПТЭ (приказ Минэнерго № 811), ПТЭЭП и ГОСТ на этапе 4 в индекс не включались.


Ячейка ниже считает объём базы и наборов вопросов по файлам репозитория (в отчёте: 44 файла DOCX, 44 файла Markdown, 43 файла чанков, 2912 чанков, из них 2838 с номером пункта и 74 без него; 165 чанков с таблицами, 184 с изображениями). Изображения: `data/chunked/image_*`, 26 каталогов, 760 файлов.

Все результаты сравнения опытов получены на итоговом наборе `eval_questions.json` (268 вопросов, Q-0560…Q-0827). Каждая запись содержит вопрос, эталонный пункт ПУЭ (`expected_clause`) и раздел. Первые прогоны (до 07.10.2026) выполнены на раннем наборе из 25 вопросов, отдельным файлом он не сохранён.

In [ ]:
chunk_dir = ROOT / "data" / "chunked"
files = sorted(chunk_dir.glob("*.chunked.jsonl")) if chunk_dir.is_dir() else []
if files:
    recs = [json.loads(l) for f in files for l in f.read_text(encoding="utf-8").splitlines() if l.strip()]
    clauses = [r["metadata"].get("Clause") for r in recs if r["metadata"].get("Clause")]
    md_table(["Показатель", "Значение"], [
        ["Файлов DOCX (ПУЭ)", len(list((ROOT / "data/raw/Нормативная база/ПУЭ/DOCX").glob("*.docx")))],
        ["Файлов Markdown", len(list((ROOT / "data/extracted").glob("*.md")))],
        ["Файлов чанков JSONL", len(files)],
        ["Чанков всего", len(recs)],
        ["Чанков с номером пункта (Clause)", f"{len(clauses)} (различных: {len(set(clauses))})"],
        ["Чанков без номера пункта", len(recs) - len(clauses)],
        ["Чанков с таблицами", sum(1 for r in recs if r["metadata"].get("contains_tables"))],
        ["Чанков с изображениями", sum(1 for r in recs if r["metadata"].get("contains_images"))],
    ])
else:
    print("Папка data/chunked не найдена — статистика базы пропущена.")

PURPOSE = {  # назначение наборов — как в таблице раздела 2 отчёта Word
    "eval_questions_base38.json": "ранний снимок",
    "eval_questions_74.json": "ранний снимок (в прогоне 73)",
    "eval_questions.json": "**итоговый рабочий набор этапа 4**",
    "eval_questions_300_final.json": "исходник итогового набора (копия eval_questions.json)",
    "eval_questions_pue_expanded_final.json": "расширенный набор",
    "eval_questions_374_final.json": "дополнительный набор",
    "eval_questions_pue_all.json": "общий реестр: объединение наборов на 559, 268 и 337 вопросов",
}
rows = [["ранний набор (отдельно не сохранён)", 25, "", "первые прогоны (до 07.10.2026)"]]
for f in sorted((ROOT / "data" / "rag_eval_questions").glob("*.json")):
    d = load(f)
    n = d.get("n_questions", len(d.get("questions", []))) if isinstance(d, dict) else len(d)
    rows.append([f"`{f.name}`", n, (d.get("registry_range", "") if isinstance(d, dict) else ""),
                 PURPOSE.get(f.name, "вспомогательный, в отчёте не используется")])
md_table(["Набор вопросов", "Вопросов", "Диапазон registry_id", "Назначение"], rows)

qs = load(QUESTIONS)["questions"]
SECTION = {q["id"]: q["section"].split(".")[0].replace("Раздел ", "Р.") for q in qs}
cnt = Counter(SECTION.values())
print(f"Итоговый рабочий набор eval_questions.json: {len(qs)} вопросов; по разделам ПУЭ:",
      ", ".join(f"{k} — {cnt[k]}" for k in sorted(cnt)))

## 3. Параметризация данных

**Сегментация (чанкинг).** Парсер `src/preprocessing/Create_chunkeds/md_to_chunked_2.py` разбирает Markdown по уровням: документ (Document), раздел (Section), глава (Chapter), параграф (Paragraph) и пункт (Clause). Правило: один пункт ПУЭ — один чанк.

Пример чанка (тот же, что в отчёте Word; ячейка ниже выводит эту запись прямо из `data/chunked/1.1.chunked.jsonl`):

```json
{
    "id": "pue_1.1.39_9ab608fe",
    "metadata": {
        "Document": "ПРАВИЛА УСТРОЙСТВА ЭЛЕКТРОУСТАНОВОК",
        "Section": "Раздел 1 ОБЩИЕ ПРАВИЛА",
        "Chapter": "",
        "Paragraph": "Общие указания по устройству электроустановок",
        "Clause": "1.1.39",
        "contains_tables": false,
        "contains_images": false,
        "source_file": "1.1.md"
    },
    "content": "ПРАВИЛА УСТРОЙСТВА ЭЛЕКТРОУСТАНОВОК. Раздел 1 ОБЩИЕ ПРАВИЛА. Общие указания по устройству электроустановок. Пункт 1.1.39.\n\n1.1.39. Вновь сооруженные и реконструированные электроустановки вводятся в промышленную эксплуатацию только после их приемки согласно действующим положениям.",
    "created_at": "2026-03-06T00:42:08.846545"
}
```

Заголовки в текст чанка не попадают; перед текстом добавляется контекстная строка «документ. раздел. глава. параграф. Пункт X.Y.Z». В метаданных сохраняются Document, Section, Chapter, Paragraph, Clause, а также признаки наличия таблиц и изображений.

In [ ]:
# Пример чанка из файла (пункт 1.1.39), как в разделе 3 отчёта
f = ROOT / "data" / "chunked" / "1.1.chunked.jsonl"
if f.is_file():
    rec = next((json.loads(l) for l in f.read_text(encoding="utf-8").splitlines()
                if l.strip() and '"pue_1.1.39_' in l), None)
    print(json.dumps(rec, ensure_ascii=False, indent=4) if rec else "Чанк 1.1.39 не найден")
else:
    print("Файл data/chunked/1.1.chunked.jsonl не найден")

**Векторизация и база** (общие для трёх опытов):

| Параметр | Значение |
|---|---|
| Модель эмбеддингов текста | `intfloat/multilingual-e5-base`, размерность 768 |
| Нормализация векторов | `normalize_embeddings=True` |
| Устройство | CUDA |
| Модель для изображений (в конфиге) | OpenCLIP `ViT-B-32` на CPU; при оценке не загружается |
| Векторная БД | Milvus standalone (Docker), `localhost:19530` |
| Индекс | HNSW по полю `text_vector`, COSINE, M=8, efConstruction=200 |
| Параметры поиска | COSINE, ef=64 |
| Размер выдачи (top-k) | 5 (`query.default_limit`) |
| Загрузка | пакеты по 32, коллекция пересоздаётся (`drop_existing`) |
| Фильтр по разделу | не используется |
| Глубина выдачи для MRR/nDCG | 10 (`k_max`, отдельный прогон 28.10.2026) |

**Языковая модель (LLM).** Модель `qwen2.5:14b-instruct-q4_K_M` запускается через Ollama с параметрами: temperature 0,2, num_predict 512 (ответ не длиннее 512 токенов), таймаут 120 с. Системный промпт требует отвечать только по приведённому контексту, при отсутствии ответа так и сказать и указывать номера пунктов ПУЭ.

**Эмбеддинг-модель (E5).** Вопрос и фрагмент кодируются по отдельности в вектор из 768 чисел, Milvus сравнивает векторы и возвращает ближайшие. При обучении модели роль текста указывали меткой в начале строки: у фрагмента документа — `passage:`, у вопроса — `query:`. В сохранённый пункт ПУЭ эти метки не записываются, они добавляются только при кодировании. В опыте 1 меток нет, в опытах 2 и 3 они есть.

**Реранкер.** Модель `BAAI/bge-reranker-v2-m3` оценивает каждую пару «вопрос — фрагмент» из уже найденных первым поиском; по всей базе она не ищет. В опыте 3 Milvus сначала отдаёт 20 кандидатов, реранкер оставляет из них 5 лучших; они идут в ответ и в подсчёт Hit@k. В опытах 1 и 2 этого шага нет.

**Параметры трёх опытов.** Модель, индекс, LLM и набор вопросов одинаковы; меняются префиксы E5 и реранкер. Каждый опыт индексирован в отдельную коллекцию. Таблица ниже читается из конфигурационных файлов:

In [ ]:
cfg_paths = {
    1: ROOT / "config" / "rag_runtime.json",
    2: ROOT / "src/preprocessing/Create_embeddings/E5/rag_runtime_e5.json",
    3: ROOT / "src/preprocessing/Create_embeddings/Rerank/rag_runtime_rerank.json",
}
cfg = {k: load(p) for k, p in cfg_paths.items()}
def g(c, *path):
    for p in path:
        c = (c or {}).get(p)
    return c
rows = []
for name, getter in [
    ("Коллекция Milvus", lambda n: g(cfg[n], "vector_db", "collection_name")),
    ("Записей в коллекции", lambda n: g(DATA[n]["hitk"], "collection_num_entities")),
    ("Префикс passage: при индексации", lambda n: "да" if n > 1 else "нет"),
    ("Префикс query: при поиске", lambda n: "да" if n > 1 else "нет"),
    ("Реранкер", lambda n: (g(cfg[n], "rerank", "model") + ", " + str(g(cfg[n], "models", "device_text")).upper()) if g(cfg[n], "rerank", "model") else "нет"),
    ("Кандидатов из Milvus", lambda n: g(cfg[n], "rerank", "candidates") or g(cfg[n], "query", "default_limit")),
    ("Фрагментов в метрику и LLM", lambda n: str(g(cfg[n], "query", "default_limit")) + (" (после реранкера)" if n == 3 else "")),
    ("Глубина для MRR/nDCG", lambda n: (DATA[n]["rank"] or {}).get("k_max", "—")),
    ("Модель эмбеддингов", lambda n: g(cfg[n], "models", "text_model_name")),
    ("Устройство", lambda n: str(g(cfg[n], "models", "device_text")).upper()),
    ("LLM (Ollama)", lambda n: g(cfg[n], "ollama", "model")),
    ("temperature / num_predict", lambda n: f'{num(g(cfg[n], "ollama", "temperature"))} / {g(cfg[n], "ollama", "num_predict")}'),
    ("Конфиг", lambda n: f"`{cfg_paths[n].relative_to(ROOT).as_posix()}`"),
]:
    rows.append([name] + [getter(n) for n in (1, 2, 3)])
md_table(["Параметр"] + LABELS, rows)

## 4. Архитектура рабочего контура (прототип RAG)

### 4.1. Общая схема

Прототип состоит из двух готовых нейросетей: модели эмбеддингов `intfloat/multilingual-e5-base` (размерность 768, расчёт на видеокарте) и языковой модели `qwen2.5:14b-instruct-q4_K_M` (Ollama, temperature 0,2, ответ не длиннее 512 токенов), векторной базы Milvus и, в опыте 3, реранкера `BAAI/bge-reranker-v2-m3` (видеокарта, из 20 кандидатов в ответ идут 5).

```
Вопрос пользователя
   │  ("query: " + вопрос — в опытах 2 и 3)
   ▼
multilingual-e5-base: вектор из 768 чисел
   ▼
Milvus (HNSW, COSINE): 5 фрагментов (опыты 1–2) или 20 кандидатов (опыт 3)
   ▼
[опыт 3] bge-reranker-v2-m3: из 20 кандидатов остаются 5
   ▼
Промпт: системная инструкция + 5 фрагментов ПУЭ + вопрос
   ▼
Ollama qwen2.5:14b: ответ с номерами пунктов ПУЭ
```

### 4.2. Реализованные модули (код)

| Компонент | Файл | Назначение |
|---|---|---|
| Подготовка чанков | `src/preprocessing/Create_chunkeds/md_to_chunked_2.py` | перевод Markdown в JSONL с метаданными пунктов |
| Класс RAG | `src/preprocessing/Create_embeddings/multimodal_rag.py` | эмбеддинги, запись и поиск в Milvus, реранкер |
| Загрузка в Milvus | `load_data.py`, `E5/load_data_e5.py`, `Rerank/load_data_rerank.py` | индексация для опытов 1, 2, 3 |
| Поиск с реранкером | `src/preprocessing/Create_embeddings/Rerank/rerank_rag.py` | открытие коллекции опыта 3 |
| Сервис RAG | `src/rag/rag_service.py` | поиск, формирование контекста и ответ (`answer()`) |
| Клиент LLM | `src/rag/ollama_client.py` | вызов API Ollama |
| Конфигурация | `config/rag_runtime.json` | адрес Milvus, модели, top-k, параметры Ollama |
| Инфраструктура | `infra/milvus/`, `infra/docker/Dockerfile.ollama/` | Docker Compose для Milvus и Ollama |

Telegram-бот (`src/bot/`) к контуру на этапе 4 не подключён.

### 4.3. Как считается метрика качества ранжированного поиска (Hit@k)

Метрика Hit@k (также Hit Rate@k, Hits@k) — это не параметр нейросети, а стандартная метрика ранжированного поиска (information retrieval): доля запросов, для которых хотя бы один релевантный документ попал в первые k позиций выдачи. В этой работе Hit@k играет роль «первой точности»: модели не дообучались на ПУЭ, поэтому графика ошибки по эпохам нет. Например, Hit@5 = 81,3 % в опыте 1 означает, что в 218 вопросах из 268 нужный пункт был среди пяти фрагментов.

Для одного запроса q: hit@k(q) = 1, если релевантный документ есть среди первых k результатов, иначе 0. По набору из N запросов:

$$\mathrm{Hit@}k = \frac{1}{N}\sum_{q} \mathrm{hit@}k(q)$$

- N = 268 — число вопросов в итоговом наборе `eval_questions.json` (поле `n_questions`).
- На каждый вопрос есть ровно один эталонный пункт, поэтому Hit@k совпадает с полнотой поиска по первым k фрагментам (Recall@k).
- На каждый вопрос выполняется один поиск; языковая модель в подсчёте Hit@k не участвует.
- Попадание фиксируется по тексту фрагмента: номер искомого пункта должен встретиться в одной из форм «п.1.7.103», «п. 1.7.103», «Пункт 1.7.103».
- Берутся k = 1, 3 и 5, поэтому всегда Hit@1 ≤ Hit@3 ≤ Hit@5.

| Метрика | Смысл в проекте |
|---|---|
| Hit@1 | Эталонный пункт оказался первым |
| Hit@3 | Эталонный пункт есть в тройке |
| Hit@5 | Эталонный пункт есть в пятёрке (лимит выдачи RAG) |

## 5. Графическое подтверждение (метрики прототипа)

- **Hit@1, Hit@3 и Hit@5** — доля вопросов, у которых нужный пункт ПУЭ оказался первым, в первой тройке или в первой пятёрке найденных фрагментов (подробно — п. 4.3).
- **Качество ответа** (полный контур: поиск пяти фрагментов и генерация текста). «Пункт найден или назван» (`citation_rate`) — доля вопросов, где номер эталонного пункта есть во фрагментах или в тексте ответа. «Номер написан в ответе» (`citation_in_answer_rate`) — в готовом ответе есть запись вида п.1.7.103. «Пункт есть в пяти фрагментах» (`citation_in_context_rate`) — номер есть в тексте, переданном модели. «Ответ из слов фрагментов» (`groundedness`, от 0 до 1) — доля содержательных предложений ответа, в которых не меньше 30 % значимых слов встречаются во фрагментах. Честный отказ «в приведённом тексте этого нет» считается опорой на текст, а не выдумкой. «Попадание в нужный пункт» (`relevance`, от 0 до 1) — средний балл по вопросам: 1, если номер написан в ответе; 0,7, если пункт есть во фрагментах и ответ не пустой; 0,3, если поиск попал в пятёрку, но номер не назван; 0,5, если модель честно написала, что во фрагментах ответа нет; иначе 0.
- **Нагрузочный тест** проверяет только языковую модель: ей отправляют короткое задание, по три запроса одновременно; поиск по ПУЭ и реранкер в тест не входят.
- **Вторая оценка той же моделью** (LLM-judge) — вспомогательная: модели дают не живой ответ бота, а шаблонный текст и шаблонные фрагменты. Три оценки от 0 до 1: «без фактов вне текста» (faithfulness), «ответ по теме вопроса» (answer relevancy), «в тексте хватает материала» (context recall).
- Насколько высоко в списке стоит нужный пункт, показывают метрики **MRR и nDCG** (п. 5.5).

### 5.1. Опыт 1 (база): серия прогонов

Чанки и вопросы кодируются без префиксов, выполняется один поиск первых пяти фрагментов. Для опыта 1 проведена серия прогонов на последовательных версиях контрольного набора: 25, 38, 50, 73 и 300 вопросов, затем итоговый прогон на 268 вопросах. Коллекция и параметры поиска во всех прогонах не менялись, менялся только набор вопросов. Источник — журнал `stage4_hitk_runs.jsonl`; подряд идущие прогоны на одном N объединены (у прогона 1.6 два запуска: 18.10.2026 16:58 и итоговый 25.10.2026 20:00, значения совпали).

In [ ]:
runs = [json.loads(l) for l in (RDIR / "stage4_hitk_runs.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
series = []  # группы подряд идущих прогонов с одинаковым N
for r in runs:
    if series and series[-1]["n"] == r["n_questions"]:
        series[-1]["runs"].append(r)
    else:
        series.append({"n": r["n_questions"], "runs": [r]})
rows = []
for i, s in enumerate(series, 1):
    first, last = s["runs"][0], s["runs"][-1]
    m = first["metrics_percent"]
    same = all(x["metrics_percent"] == m for x in s["runs"])
    rows.append([f"1.{i}" + (" (итоговый)" if i == len(series) else ""), s["n"], first["evaluated_at"] + ("" if len(s["runs"]) == 1 else f" … {last['evaluated_at']}"),
                 len(s["runs"]), num(m["hit@1"]) + " %", num(m["hit@3"]) + " %", num(m["hit@5"]) + " %",
                 "да" if same else "нет"])
md_table(["№", "N", "Дата (первый … последний)", "Прогонов", "Hit@1", "Hit@3", "Hit@5", "Повторы совпали"], rows)

line_chart(
    [f"N={s['n']}" for s in series],
    [(k.replace("hit", "Hit"), [s["runs"][0]["metrics_percent"][k] for s in series])
     for k in ("hit@1", "hit@3", "hit@5")],
    "Рисунок 1 — Опыт 1: Hit@k в серии прогонов на разных версиях набора вопросов", "%", ymax=100,
)

Падение в прогоне 1.5 (300 вопросов) объясняется составом набора (в нём была автоматическая разметка эталонных пунктов), а не работой поиска. В этом прогоне доля «пункт найден или назван» составила 28,0 %, «попадание в нужный пункт» — 0,317. После проверки набор сокращён до 268 вопросов. Итоговый поиск опыта 1 выполнен 25.10.2026 в 20:00 (коллекция из 2912 записей, эмбеддинги на видеокарте) и дал те же значения, что прогон 1.6. Ячейка ниже считает среднюю близость лучшего фрагмента и число промахов Hit@5 по каждому опыту. Генерация ответа опыта 1 повторена на всех 268 вопросах 04.11.2026; её результаты — в п. 5.6.

In [ ]:
# Средняя близость (cosine) лучшего фрагмента: при попадании в пятёрку и при промахе; промахи Hit@5
rows = []
for n in (1, 2, 3):
    det = DATA[n]["hitk"]["details"]
    hit = [d["top1_score"] for d in det if d["hit"]["@5"] and d.get("top1_score") is not None]
    miss = [d["top1_score"] for d in det if not d["hit"]["@5"] and d.get("top1_score") is not None]
    rows.append([LABELS[n - 1], DATA[n]["hitk"]["evaluated_at"], DATA[n]["hitk"]["collection_num_entities"],
                 num(np.mean(hit), 3) if hit else "—", num(np.mean(miss), 3) if miss else "—", len(miss)])
md_table(["Опыт", "Прогон Hit@k", "Записей", "Близость, пункт в пятёрке", "Близость, промах", "Промахов Hit@5"], rows)
print("В опытах 1 и 2 близость лучшего фрагмента почти одинакова при попадании и при промахе: по ней одной промах от попадания не отличить. "
      "В опыте 3 в поле top1_score записана оценка реранкера, а не косинусная близость Milvus, поэтому её с опытами 1 и 2 не сравнивают (в отчёте Word она не приводится).")

### 5.2. Опыт 2 (префиксы E5)

Для модели E5 добавлены префиксы `passage:` при индексации и `query:` при поиске, коллекция `diplom_multimodal_e5` собрана заново. Полный прогон на 268 вопросах — 25.10.2026 (21:04–22:31), сбоев при генерации ответа нет. Результаты — в пп. 5.4–5.7.

### 5.3. Опыт 3 (E5 и реранкер)

Префиксы те же, что в опыте 2. Milvus возвращает 20 кандидатов, реранкер `BAAI/bge-reranker-v2-m3` оценивает каждую пару «вопрос — фрагмент» и переставляет кандидатов; в показатели и в модель передаются 5 лучших. Hit@k считается уже после реранкера. Полный прогон на 268 вопросах — 26.10.2026 (00:40–02:22), сбоев при генерации ответа нет. Вторая оценка не выполнилась на одном вопросе (id 178, ошибка 500 сервера Ollama); ему записаны нули, и они входят в средние значения п. 5.7.

### 5.4. Сравнение опытов: Hit@k

In [ ]:
rows = []
for k in ["hit@1", "hit@3", "hit@5"]:
    rows.append([k.replace("hit", "Hit")] + [
        f'{num(DATA[n]["hitk"]["metrics_percent"][k])} % ({sum(d["hit"]["@" + k[4:]] for d in DATA[n]["hitk"]["details"])}/{DATA[n]["hitk"]["n_questions"]})'
        for n in (1, 2, 3)])
MISS5 = {n: {d["id"] for d in DATA[n]["hitk"]["details"] if not d["hit"]["@5"]} for n in (1, 2, 3)}
rows.append(["Промахов Hit@5"] + [len(MISS5[n]) for n in (1, 2, 3)])
rows.append(["Дата прогона"] + [DATA[n]["hitk"]["evaluated_at"] for n in (1, 2, 3)])
md_table(["Метрика"] + LABELS, rows)
bar_chart(["Hit@1", "Hit@3", "Hit@5"],
          [(LABELS[n - 1], [DATA[n]["hitk"]["metrics_percent"][k] for k in ("hit@1", "hit@3", "hit@5")]) for n in (1, 2, 3)],
          "Рисунок 2 — Hit@k по трём опытам (N = 268)", "%", ymax=100)
hp = {n: DATA[n]["hitk"]["metrics_percent"] for n in (1, 2, 3)}
d = lambda a, b, k: num(hp[a][k] - hp[b][k]) + " п.п."
print(f"Префиксы E5: Hit@1 +{d(2,1,'hit@1')}, Hit@5 +{d(2,1,'hit@5')}. Реранкер относительно опыта 2: Hit@1 +{d(3,2,'hit@1')}, "
      f"Hit@5 +{d(3,2,'hit@5')}; относительно опыта 1: Hit@1 +{d(3,1,'hit@1')}, Hit@5 +{d(3,1,'hit@5')} "
      "(разность округлённых процентов).")
print(f"Опыт 2 исправил {len(MISS5[1] - MISS5[2])} промахов Hit@5 опыта 1 и добавил {len(MISS5[2] - MISS5[1])}; "
      f"опыт 3 исправил {len(MISS5[2] - MISS5[3])} промахов опыта 2 и добавил {len(MISS5[3] - MISS5[2])}; "
      f"не найдено ни в одном опыте: {len(MISS5[1] & MISS5[2] & MISS5[3])}.")

Метрика Hit@5 по разделам ПУЭ (подсчитано по тем же JSON):

In [ ]:
secs = sorted(set(SECTION.values()))
tab = {}
for n in (1, 2, 3):
    tot, hit = Counter(), Counter()
    for d in DATA[n]["hitk"]["details"]:
        s = SECTION[d["id"]]; tot[s] += 1; hit[s] += d["hit"]["@5"]
    tab[n] = {s: 100 * hit[s] / tot[s] for s in secs}
md_table(["Раздел", "Вопросов"] + LABELS, [[s, cnt[s]] + [num(tab[n][s]) + " %" for n in (1, 2, 3)] for s in secs])
bar_chart(secs, [(LABELS[n - 1], [tab[n][s] for s in secs]) for n in (1, 2, 3)],
          "Рисунок 3 — Hit@5 по разделам ПУЭ для трёх опытов", "%", ymax=110, fmt="{:.0f}")
print(f"Самый слабый раздел во всех опытах — раздел 2. В разделе 5 реранкер снизил результат "
      f"(с {round(tab[2]['Р.5'] * cnt['Р.5'] / 100)} до {round(tab[3]['Р.5'] * cnt['Р.5'] / 100)} попаданий из {cnt['Р.5']}).")

### 5.5. Сравнение опытов: MRR и nDCG@k

Метрика Hit@5 не различает, стоит нужный пункт первым или пятым. Чтобы измерить само место, по тем же 268 вопросам построен список из 10 фрагментов (отдельный прогон 28.10.2026); номер пункта проверяется так же, как в Hit@k.

**MRR** (Mean Reciprocal Rank, средний обратный ранг): для каждого вопроса берётся 1/место нужного пункта (первое место — 1, второе — 1/2, третье — 1/3; если в десятке пункта нет — 0), затем среднее по всем вопросам. MRR@3 и MRR@5 считаются так же, но место ниже третьего или пятого даёт 0.

**nDCG@5** (нормированная дисконтированная полезность) тоже учитывает место, но по первым пяти позициям и с более плавным снижением: вклад вопроса равен 1/log₂(место + 1). Первое место даёт 1, второе — 0,63, третье — 0,50, пятое — 0,39; если пункта нет в пятёрке — 0. nDCG@3 и nDCG@10 — то же по первым трём и первым десяти местам.

В таблице ниже названия строк те же, что в отчёте Word («Штраф за дальнее место» — это nDCG@k).

In [ ]:
RANK_KEYS = ["hit@10", "mrr", "mrr@3", "mrr@5", "ndcg@3", "ndcg@5", "ndcg@10"]
RNAME = {"hit@10": "Пункт нашёлся в десятке (Hit@10)", "mrr": "Среднее 1/место (MRR)", "mrr@3": "Среднее 1/место, первые 3 (MRR@3)", "mrr@5": "Среднее 1/место, первые 5 (MRR@5)", "ndcg@3": "Штраф за дальнее место, первые 3 (nDCG@3)", "ndcg@5": "Штраф за дальнее место, первые 5 (nDCG@5)", "ndcg@10": "Штраф за дальнее место, первые 10 (nDCG@10)"}
RSHORT = {"hit@10": "Hit@10", "mrr": "MRR", "mrr@3": "MRR@3", "mrr@5": "MRR@5", "ndcg@3": "nDCG@3", "ndcg@5": "nDCG@5", "ndcg@10": "nDCG@10"}
if all(DATA[n]["rank"] for n in (1, 2, 3)):
    rows = [[RNAME[k]] + [num(DATA[n]["rank"]["metrics"][k], 3) for n in (1, 2, 3)] for k in RANK_KEYS]
    rows.append(["Дата прогона"] + [DATA[n]["rank"]["evaluated_at"] for n in (1, 2, 3)])
    md_table(["Метрика"] + LABELS, rows)
    for n in (1, 2, 3):  # проверка: Hit@1/3/5 в rank-прогоне = основной прогон
        for k in ("hit@1", "hit@3", "hit@5"):
            a, b = DATA[n]["hitk"]["metrics_percent"][k], round(DATA[n]["rank"]["metrics"][k] * 100, 1)
            if abs(a - b) > 0.05:
                print(f"! {LABELS[n-1]}: {k} {a} ≠ {b}")
    bar_chart([RSHORT[k] for k in RANK_KEYS], [(LABELS[n - 1], [DATA[n]["rank"]["metrics"][k] for k in RANK_KEYS]) for n in (1, 2, 3)],
              "Рисунок 4 — MRR и nDCG@k по трём опытам (N = 268)", "0–1", ymax=1.05, fmt="{:.2f}")
    r = {n: DATA[n]["rank"]["metrics"] for n in (1, 2, 3)}
    print(f"Префиксы E5: MRR +{num(r[2]['mrr'] - r[1]['mrr'], 3)}; реранкер: MRR +{num(r[3]['mrr'] - r[2]['mrr'], 3)} относительно опыта 2, "
          f"nDCG@5 +{num(r[3]['ndcg@5'] - r[2]['ndcg@5'], 3)}. В выдаче глубиной 10 опыт 3 находит эталонный пункт в "
          f"{num(r[3]['hit@10'] * 100)} % вопросов ({round(r[3]['hit@10'] * 268)} из 268).")
else:
    print("Нет stage4_rank_results_*.json — запустите run_rank_all.ps1 (раздел 6).")

### 5.6. Сравнение опытов: качество ответа

Числа берутся только из JSON. Если файл посчитан не на всех вопросах (`partial_run` или N ≠ 268), его значения в таблицу и графики не попадают. Названия показателей — как в отчёте Word, в скобках поле JSON.

In [ ]:
GEN_KEYS = ["citation_rate", "citation_in_answer_rate", "citation_in_context_rate", "groundedness_avg", "relevance_avg"]

GEN, GEN_SRC = {}, {}
for n, label, suf, mode in EXPERIMENTS:
    g_ = DATA[n]["gen"]; N = DATA[n]["hitk"]["n_questions"]
    m = (g_ or {}).get("metrics", {})
    if g_ is not None and not m.get("partial_run") and m.get("n_questions") == N:
        GEN[n] = dict(m); GEN[n]["errors"] = sum(1 for d in g_["details"] if d.get("error"))
        GEN_SRC[n] = f"stage4_gen_eval_results{suf}.json ({g_['evaluated_at']})"
    else:
        GEN[n] = {}
        GEN_SRC[n] = "нет полного прогона"
GNAME = {"citation_rate": "Пункт найден или назван, % (citation_rate)",
         "citation_in_answer_rate": "Номер написан в ответе, % (citation_in_answer_rate)",
         "citation_in_context_rate": "Пункт есть в пяти фрагментах, % (citation_in_context_rate)",
         "groundedness_avg": "Ответ из слов фрагментов, 0–1 (groundedness)",
         "relevance_avg": "Попадание в нужный пункт, 0–1 (relevance)"}
rows = [[GNAME[k]] + [num(GEN[n].get(k), 1 if k.endswith("rate") else 3) + (" %" if k.endswith("rate") and GEN[n].get(k) is not None else "") for n in (1, 2, 3)] for k in GEN_KEYS]
rows.append(["Среднее время полного ответа, с"] + [num(GEN[n].get("duration_avg_sec"), 2) for n in (1, 2, 3)])
rows.append(["Сбоев при генерации"] + [GEN[n].get("errors", "нет данных") for n in (1, 2, 3)])
rows.append(["Источник (дата прогона)"] + [GEN_SRC[n] for n in (1, 2, 3)])
md_table(["Метрика"] + LABELS, rows)
bar_chart(["Пункт найден или назван", "Номер написан в ответе"],
          [(LABELS[n - 1], [GEN[n].get("citation_rate"), GEN[n].get("citation_in_answer_rate")]) for n in (1, 2, 3)],
          "Рисунок 5 — Пункт найден или номер написан в ответе", "%", ymax=100)
bar_chart(["Ответ из слов фрагментов", "Попадание в нужный пункт"],
          [(LABELS[n - 1], [GEN[n].get("groundedness_avg"), GEN[n].get("relevance_avg")]) for n in (1, 2, 3)],
          "Рисунок 6 — Ответ из слов фрагментов и попадание в нужный пункт", "0–1", ymax=1.05, fmt="{:.3f}")
bar_chart(["Полный ответ"], [(LABELS[n - 1], [GEN[n].get("duration_avg_sec")]) for n in (1, 2, 3)],
          "Рисунок 7 — Среднее время запроса полного RAG по опытам", "с", fmt="{:.2f}")
if all(GEN[n] for n in (1, 2, 3)):
    print(f"Доля «пункт найден или назван» во всех опытах совпала с Hit@5: модель не пишет номер, которого не было среди пяти фрагментов. "
          f"В самом тексте ответа номер встречается реже — в {num(GEN[1]['citation_in_answer_rate'])}–{num(GEN[3]['citation_in_answer_rate'])} % вопросов. "
          f"Реранкер увеличил среднее время полного ответа на {num(GEN[3]['duration_avg_sec'] - GEN[2]['duration_avg_sec'], 2)} с по сравнению с опытом 2.")

Время одного поиска без языковой модели измерено в прогоне MRR/nDCG 28.10.2026 по каждому из 268 вопросов:

In [ ]:
def durs(d):
    return [x.get("duration_sec") for x in (d or {}).get("details", []) if x.get("duration_sec") is not None]
TIME = {}
for n in (1, 2, 3):
    xs = durs(DATA[n]["rank"])
    TIME[n] = {"rag": GEN[n].get("duration_avg_sec"),
               "retr": sum(xs) / len(xs) if xs else None,
               "retr_med": float(np.median(xs)) if xs else None,
               "first": xs[0] if xs else None}
md_table(["Время поиска, с"] + LABELS, [
    ["Среднее"] + [num(TIME[n]["retr"], 3) for n in (1, 2, 3)],
    ["Медиана"] + [num(TIME[n]["retr_med"], 3) for n in (1, 2, 3)],
    ["Первый запрос (холодный старт)"] + [num(TIME[n]["first"], 2) for n in (1, 2, 3)],
])
bar_chart(["Только поиск"], [(LABELS[n - 1], [TIME[n]["retr"]]) for n in (1, 2, 3)],
          "Рисунок 8 — Среднее время только поиска по опытам", "с", fmt="{:.3f}")
print(f"Поиск в Milvus занимает около {num(TIME[1]['retr'], 2)} с на вопрос, реранкер добавляет около {num(TIME[3]['retr'] - TIME[2]['retr'], 1)} с. "
      f"Большая часть времени полного ответа ({num(TIME[1]['rag'] + 1e-9, 1)}–{num(TIME[3]['rag'] + 1e-9, 1)} с) уходит на генерацию текста языковой моделью.")

In [ ]:
PK = [("ttft_avg_sec", "До первого слова, с (TTFT)"), ("latency_avg_sec", "Полное время генерации, с (latency avg)"),
      ("latency_p95_sec", "Не превышают 95 % запросов, с (p95)"), ("latency_p99_sec", "Не превышают 99 % запросов, с (p99)"),
      ("tps_avg", "Токенов в секунду (TPS)")]
rows = [[name] + [num(DATA[n]["perf"]["metrics"][k], 2) for n in (1, 2, 3)] for k, name in PK]
rows += [["Запросов в секунду (RPS)"] + [num(DATA[n]["perf"]["rps"], 2) for n in (1, 2, 3)],
         ["Доля ответивших запросов"] + [f'{num(DATA[n]["perf"]["availability_pct"], 0)} % ({DATA[n]["perf"]["successful_requests"]}/{DATA[n]["perf"]["total_requests"]})' for n in (1, 2, 3)]]
md_table(["Нагрузочный тест языковой модели"] + LABELS, rows)
bar_chart(["До первого слова", "Полное время генерации"],
          [(LABELS[n - 1], [DATA[n]["perf"]["metrics"][k] for k, _ in PK[:2]]) for n in (1, 2, 3)],
          "Рисунок 9 — Нагрузочный тест: время до первого слова и полное время генерации", "с", fmt="{:.2f}")

JK = [("faithfulness_avg", "Без фактов вне текста, 0–1 (faithfulness)"), ("answer_relevancy_avg", "Ответ по теме вопроса, 0–1 (answer relevancy)"), ("context_recall_avg", "В тексте хватает материала, 0–1 (context recall)")]
JSHORT = ["Без фактов вне текста", "Ответ по теме вопроса", "В тексте хватает материала"]
JFAIL = {n: [d["id"] for d in DATA[n]["judge"]["details"] if "Ошибка вызова" in str((d.get("scores") or {}).get("reasoning", ""))] for n in (1, 2, 3)}
rows = [[name] + [num(DATA[n]["judge"]["summary_metrics"][k], 4) for n in (1, 2, 3)] for k, name in JK]
rows.append(["Оценено без сбоя"] + [f'{DATA[n]["judge"]["sample_count"] - len(JFAIL[n])}/{DATA[n]["judge"]["sample_count"]}'
                                     + (" (сбой id=" + ", ".join(map(str, JFAIL[n])) + ")" if JFAIL[n] else "") for n in (1, 2, 3)])
md_table(["Вторая оценка той же моделью"] + LABELS, rows)
bar_chart(JSHORT, [(LABELS[n - 1], [DATA[n]["judge"]["summary_metrics"][k] for k, _ in JK]) for n in (1, 2, 3)],
          "Рисунок 10 — Вторая оценка той же моделью", "0–1", ymax=0.7, fmt="{:.3f}")
for n in (1, 2, 3):
    if JFAIL[n]:
        ok = [d["scores"] for d in DATA[n]["judge"]["details"] if d["id"] not in JFAIL[n]]
        keys = [("faithfulness", "faithfulness_avg"), ("answer_relevancy", "answer_relevancy_avg"), ("context_recall", "context_recall_avg")]
        vals = []
        for k, ks in keys:
            xs = [s.get(k) for s in ok if isinstance(s.get(k), (int, float))]
            vals.append(num(sum(xs) / len(xs), 4) if xs else "—")
        print(f"{LABELS[n-1]}: средние занижены нулями вопроса {JFAIL[n]}; без него оценки равны {', '.join(vals)}.")

Нагрузочный тест не вызывает поиск, поэтому различия между опытами отражают колебания загрузки видеокарты, а не влияние префиксов или реранкера. Вторая оценка тоже почти не различается: модели каждый раз дают один и тот же шаблонный текст, а не ответ конкретного прогона.

### 5.8. Типовые отказы поиска и ответа

1. Набор вопросов не покрывает документ. На этапе 4 это решено набором из 268 вопросов по всем семи разделам ПУЭ.
2. Поиск не принёс нужный пункт в пять фрагментов. В опыте 3 таких вопросов 30, и 23 из них не нашлись ни в одном опыте.
3. Нужный пункт найден, но стоит ниже общего определения. Реранкер поднял долю первого места с 60,1 % до 72,0 %.
4. В пятёрку попадают соседние пункты, и модели труднее назвать точный номер.
5. Модель дописывает числа и требования, которых нет во фрагментах, хотя в задании сказано отвечать только по приведённому тексту.
6. Ответ не успевает за отведённое время. В настройках предел ожидания — 120 с.
7. Не запущены Milvus или Docker. Проверка: `python scripts/infra_start_system/start_milvus_logs.py`.

## 6. Ноутбук или .py файл (прототип)

Основной ноутбук — этот файл, `Этапы/Reports/etap4/Etap4_final.ipynb`. В нём те же разделы, что в отчёте, а таблицы и графики раздела 5 строятся по сохранённым файлам результатов. Milvus и Ollama для этого не нужны: достаточно открыть ноутбук и выполнить все ячейки (Run All). Повторный прогон оценки по умолчанию выключен (`RUN_LIVE = False`).

Пример задания для модели: «Отвечай только по приведённым фрагментам ПУЭ. Если ответа во фрагментах нет — так и напиши. Указывай номер пункта в виде п.1.7.103». В запрос подставляются пять найденных фрагментов и вопрос пользователя. Вызов: `POST http://localhost:11434/api/generate`, поля `model`, `prompt`, `stream: false`.

Сводку по трём опытам собирает скрипт `scripts/stage4_eval/reports/build_stage4_summary.py`: он читает файлы результатов и проверяет, что каждый прогон выполнен на всех вопросах. Скрипты оценки (`scripts/stage4_eval/`):

| Файл | Назначение |
|---|---|
| `retrieval/eval_retrieval_hitk.py` | Hit@k, опыт 1 |
| `retrieval/eval_retrieval_hitk_e5.py` | Hit@k, опыт 2 (префиксы E5) |
| `retrieval/eval_retrieval_hitk_rerank.py` | Hit@k, опыт 3 (реранкер) |
| `retrieval/plot_stage4_hitk.py` | графики Hit@k (PNG) |
| `retrieval/stage4_hitk_report.py`, `stage4_hitk_notebook.py` | таблицы и ячейки отчёта по JSON Hit@k |
| `generation/eval_rag_generation.py` | полный ответ: нашёлся ли пункт, написан ли номер, опора на фрагменты, время |
| `generation/eval_rag_performance.py` | нагрузочный тест Ollama |
| `generation/eval_rag_llm_judge.py` | вторая оценка той же моделью |
| `demo/demo_stage4_milvus_search.py` | демо поиска в Milvus |
| `retrieval/mrr_ndcg/eval_retrieval_rank.py` | MRR, MRR@k, nDCG@k, Hit@10 (`--mode base/e5/rerank`, глубина 10) |
| `retrieval/mrr_ndcg/rank_metrics.py` | расчёт ранговых метрик и сводная таблица `rank_metrics_comparison.md/.csv`; режим `--from-hitk` — границы по Hit@k |
| `retrieval/mrr_ndcg/test_rank_metrics.py` | тесты формул |
| `run_rank.ps1`, `run_rank_all.ps1`, `run_rank_metrics.ps1` | MRR/nDCG для одного опыта / всех трёх / только сводная таблица |
| `reports/build_stage4_summary.py` | сводные таблицы по трём опытам |
| `run_all.ps1`, `run_all_e5.ps1`, `run_all_rerank.ps1` | полный прогон опыта 1 / 2 / 3: инфраструктура, Hit@k, место в списке, ответ, нагрузка, вторая оценка |
| `run_load_rerank.ps1` | загрузка коллекции опыта 3 |

Флаги `run_all*.ps1`: `-MilvusOnlyHitk` — только Hit@k; `-SkipPerfJudge` — без нагрузочного теста и второй оценки. Результаты пишутся в `scripts/stage4_eval/results/` (папки `retrieval` и `generation`), журналы — в `results/logs/`.

Ячейка ниже проверяет наличие скриптов. Следующая — повторный прогон (Docker, Milvus, Ollama, Windows/PowerShell); выполняется только при `RUN_LIVE = True` (ячейка раздела 0) и перезаписывает файлы результатов.

In [ ]:
SCRIPTS = ["retrieval/eval_retrieval_hitk.py", "retrieval/eval_retrieval_hitk_e5.py", "retrieval/eval_retrieval_hitk_rerank.py",
           "retrieval/mrr_ndcg/eval_retrieval_rank.py", "retrieval/mrr_ndcg/rank_metrics.py",
           "generation/eval_rag_generation.py", "generation/eval_rag_performance.py", "generation/eval_rag_llm_judge.py",
           "retrieval/plot_stage4_hitk.py", "retrieval/stage4_hitk_report.py", "retrieval/stage4_hitk_notebook.py",
           "retrieval/mrr_ndcg/test_rank_metrics.py",
           "reports/build_stage4_summary.py", "demo/demo_stage4_milvus_search.py",
           "run_all.ps1", "run_all_e5.ps1", "run_all_rerank.ps1", "run_rank.ps1", "run_rank_all.ps1",
           "run_rank_metrics.ps1", "run_load_rerank.ps1"]
md_table(["Скрипт", "Есть"], [[f"`scripts/stage4_eval/{s}`", "да" if (ROOT / "scripts/stage4_eval" / s).is_file() else "НЕТ"] for s in SCRIPTS])
boot = ROOT / "scripts" / "stage4_eval" / "lib" / "_bootstrap.py"
paths = ROOT / "scripts" / "lib" / "report_paths.py"
print("scripts/stage4_eval/lib/_bootstrap.py:", "есть" if boot.is_file() else "НЕТ")
print("scripts/lib/report_paths.py:", "есть" if paths.is_file() else "НЕТ")

In [ ]:
# ПОВТОРНЫЙ ПРОГОН ОЦЕНКИ — только при RUN_LIVE = True (нужны Docker, Milvus, Ollama, scripts/stage4_eval/lib/_bootstrap.py).
import subprocess
if not RUN_LIVE:
    print("RUN_LIVE = False — повторный прогон пропущен, используются сохранённые результаты.")
else:
    steps = [
        ["powershell", "-ExecutionPolicy", "Bypass", "-File", r"scripts\stage4_eval\run_all.ps1"],
        ["powershell", "-ExecutionPolicy", "Bypass", "-File", r"scripts\stage4_eval\run_all_e5.ps1"],
        ["powershell", "-ExecutionPolicy", "Bypass", "-File", r"scripts\stage4_eval\run_all_rerank.ps1"],
        ["powershell", "-ExecutionPolicy", "Bypass", "-File", r"scripts\stage4_eval\run_rank_all.ps1", "-SkipInfra"],
    ]
    for cmd in steps:
        print(">", " ".join(cmd))
        rc = subprocess.run(cmd, cwd=str(ROOT)).returncode
        if rc != 0:
            raise RuntimeError(f"Шаг завершился с кодом {rc}")
    print("Готово. Перезапустите ноутбук (Run All), чтобы перечитать результаты.")

## 7. Выводы

In [ ]:
text = """
1. Прототип рабочего контура по ПУЭ собран и проверен на одном наборе из 268 вопросов: вопрос превращается в вектор, в Milvus ищутся фрагменты, Ollama пишет ответ по этим фрагментам. Языковая модель ответила на все 268 вопросов в каждом опыте, сбоев генерации не было.
2. Ранние прогоны опыта 1 на наборах разного размера и состава (от 25 до 300 вопросов) дали Hit@5 от 28,0 % до 81,3 %: результат зависел прежде всего от списка вопросов. Поэтому три опыта сравниваются только на одном фиксированном наборе из 268 вопросов.
3. Опыт 1 (без префиксов E5 и без реранкера): Hit@1 = 58,2 % (156 из 268), Hit@5 = 81,3 % (218 из 268). В 156 вопросах нужный пункт стоит первым среди фрагментов, которые получает модель, ещё в 62 — в пятёрке, но не первым. В остальных 50 вопросах нужного пункта среди фрагментов нет, и модель отвечает по чужим пунктам. Для нормативного документа это существенно: если во фрагменты попали соседние пункты, формулировку ответа в основном определяет тот, что стоит выше.
4. Опыт 2 добавляет префиксы passage: и query: к той же модели эмбеддингов. Hit@1 = 60,1 % (161 из 268), Hit@5 = 83,6 % (224 из 268): на 5 вопросов больше на первом месте и на 6 — в пятёрке. Время поиска осталось около 0,05 с; среднее время полного ответа — 6,91 с против 6,35 с в опыте 1 (разница приходится на генерацию, а не на поиск). MRR вырос с 0,681 до 0,696. Префиксы дают небольшой прирост, поэтому опыт 2 взят за основу для проверки реранкера.
5. Опыт 3 сохраняет префиксы и меняет отбор фрагментов: Milvus возвращает 20 кандидатов, реранкер bge-reranker-v2-m3 оставляет из них 5 лучших. Hit@1 = 72,0 % (193 из 268), Hit@5 = 88,8 % (238 из 268). По сравнению с опытом 2 это на 32 вопроса больше на первом месте и на 14 — в пятёрке (19 прежних промахов исправлены, 5 новых появились). В разделе 5 Hit@5 снизился с 97,4 % до 92,3 % (с 38 до 36 вопросов из 39). Доля «пункт найден или назван» (88,8 %) совпадает с Hit@5, поэтому отдельным преимуществом не считается. Оценка «попадание в нужный пункт» (0,839) тоже в основном повторяет поиск: 1 ставится, если номер написан в ответе, 0,7 — если пункт есть во фрагментах и ответ не пустой. Среднее время полного ответа — 10,15 с против 6,91 с в опыте 2. Из добавившихся 3,24 с на реранкер приходится около 0,9 с (среднее время поиска 0,93 с, медиана 0,69 с против 0,05 с в опыте 2), остальное — генерация и обмен между сервисами. Конфигурация оправдана, если важнее поставить верный пункт первым, а ответ за 10 с допустим.
6. Метрики места в списке из 10 фрагментов показывают не только факт попадания в пятёрку, но и то, насколько высоко стоит нужный пункт. В опытах 1, 2 и 3 MRR равен 0,681, 0,696 и 0,793, nDCG@5 — 0,708, 0,726 и 0,812. Основной сдвиг даёт опыт 3: Hit@1 вырос на 11,9 п.п., а Hit@5 за тот же шаг — только на 5,2 п.п., то есть реранкер прежде всего поднимает нужный пункт на первое место. Рост Hit@5 возможен только потому, что реранкер выбирает из 20 кандидатов: внутри одной и той же пятёрки он лишь меняет порядок.
7. Генерация не полностью использует то, что нашёл поиск. Доля «пункт найден или назван» совпадает с Hit@5 (81,3 %, 83,6 % и 88,8 %): модель не называет номер, которого не было во фрагментах. Доля предложений, собранных из слов фрагментов, — 0,893 в опыте 1 и 0,908 в опытах 2 и 3. Но номер вида п.1.7.103 прямо написан в ответе лишь в 69,8 %, 72,0 % и 74,6 % вопросов. В опыте 3 разрыв составляет 14,2 п.п.: нужный фрагмент у модели есть, а номера в ответе нет. Вторая оценка той же моделью почти не различается между опытами и ставится по шаблонному тексту, а не по живому ответу, поэтому конфигурацию по ней не выбирали.
8. Для дальнейшей работы выбрана конфигурация опыта 3: у неё лучшие Hit@1 и MRR, а цена по времени известна. Дальше два направления. Поиск: разобрать 30 промахов Hit@5 опыта 3 (23 из них не нашлись ни в одном опыте), проверить разметку вопросов и правила разбиения текста на фрагменты. Ответ: доработать задание модели, чтобы номер вида п.1.7.103 всегда писался в тексте, и перевести вторую оценку на живые ответы и фрагменты конкретного прогона. Рост Hit@5 сам по себе долю явных номеров в ответе (сейчас 74,6 %) не поднимает.
"""
display(Markdown(text))


## 8. План дальнейшей работы

1. Подключить RAG (конфигурация опыта 3) к Telegram-боту (`src/bot/`).
2. Поправить задание модели, чтобы в ответе всегда был номер вида п.1.7.103; вторую оценку перевести на живой ответ и фрагменты конкретного прогона.
3. Измерить задержку полного контура RAG, собрать бот, Milvus и Ollama в один Docker Compose (этап 5) и повторить оценку.

## Приложения

**Соответствие нумерации:** опыт 1 = опыт 0 в файлах проекта, опыт 2 = опыт 1 (E5), опыт 3 = опыт 2 (реранкер).

| Что | Файл |
|---|---|
| Итоговый набор вопросов (268) | `data/rag_eval_questions/eval_questions.json` |
| Hit@k, опыты 1 / 2 / 3 | `scripts/stage4_eval/results/retrieval/stage4_hitk_results.json`, `..._e5.json`, `..._rerank.json` |
| MRR/nDCG, опыты 1 / 2 / 3 | `scripts/stage4_eval/results/retrieval/stage4_rank_results_base.json`, `..._e5.json`, `..._rerank.json`, сводка `rank_metrics_comparison.md` |
| История прогонов опыта 1 | `scripts/stage4_eval/results/retrieval/stage4_hitk_runs.jsonl`, `Этапы/Reports/etap4/Etap4-hitk-svod.md` |
| Генерация ответа, нагрузка и вторая оценка | `scripts/stage4_eval/results/generation/` (без суффикса, `_e5`, `_rerank`) |
| Паспорта прогонов | `Этапы/Reports/etap4/Etap4-hitk-otchet-N268.md` (опыт 1), `Etap4-opyt1-e5-N268.md` (опыт 2), `Etap4-opyt2-rerank-N268.md` (опыт 3) |
| Описание наборов вопросов | `data/rag_eval_questions/README.md` |
| Техническое описание прогонов | `Этапы/Reports/etap4/Readme-4.md` |
| Итоговый ноутбук | `Этапы/Reports/etap4/Etap4_final.ipynb` |
| Сводка по трём опытам | `scripts/stage4_eval/reports/build_stage4_summary.py` |

### Источники по показателю Hit@k

1. Manning C. D., Raghavan P., Schütze H. Introduction to Information Retrieval. Cambridge University Press, 2008. Глава 8 — полнота поиска по первым k документам.
2. Karpukhin V. et al. Dense Passage Retrieval for Open-Domain Question Answering. EMNLP, 2020 — доля вопросов, у которых нужный фрагмент нашёлся в top-k.
3. Lewis P. et al. Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks. NeurIPS, 2020 — качество ответа зависит от того, нашёлся ли нужный фрагмент.